## 0. Project About

- Dataset: CheXpert Plus.
- Variabel Masukan: Citra Chest X-Ray.
- Variabel Target: `Findings` dan `Impression`.
- Paradigma Pembelajaran: *Supervised Learning*.
- Tugas Spesifik Pembelajaran Mesin: VLM $-$ *Medical Report Generation*.
* Permasalahan utama VLM medis saat ini:
  * kebutuhan komputasi tinggi.
  * risiko inkonsistensi klinis.
  * error propagation pada pendekatan single-sequence autoregressive.
* Penelitian mengusulkan arsitektur Decoupled Dual-Head Decoder berbasis Small Language Model (SLM) dengan optimasi QLoRA 4-bit.
* Mekanisme dual-head:
  * Head Findings menghasilkan deskripsi detail berdasarkan fitur visual citra.
  * Head Impression menghasilkan kesimpulan diagnosis berdasarkan fitur visual dan findings.
* Dataset menggunakan CheXpert Plus subset data.
* Komponen model:
  * Vision Encoder: BiomedCLIP (frozen weights).
  * Language Decoder: Qwen2.5-7B-Instruct dengan dual-LoRA adapters.

```             
             [ Citra Chest X-Ray Pasien ]
                            │
                            ▼
           ┌─────────────────────────────────┐
           │      BiomedCLIP Vision ViT      │
           │  ├─ Layer Awal-Mid: FROZEN      │
           │  └─ 4 Layer Akhir : TRAINABLE   │ ◄── Semi-Freeze (Adaptasi Visual)
           └────────────────┬────────────────┘
                            │ [Output: 512 Dim]
                            ▼
           ┌─────────────────────────────────┐
           │     Linear Projection Layer     │ ◄── TRAINABLE (Jembatan Dimensi)
           │     (512 Dim ──► 1532 Dim)      │
           └────────────────┬────────────────┘
                            │ [Visual Tokens]
                            ▼
               [ Penyambungan / Concat ] ◄─────── [ Input Prompt + History ]
                            │                     Tokenized via Qwen (3584 Dim)
                            ▼
           ┌─────────────────────────────────┐
           │     Qwen2.5-7B Base Model       │ ◄── Kuantisasi 4-bit NF4
           │   (Frozen Base Parameters)      │
           └────────────────┬────────────────┘
                            │
         ┌──────────────────┴──────────────────┐
         ▼ (Jalur Logika Aktif)                ▼ (Jalur Logika Aktif)
  ==============================        ==============================
   HEAD 1: lora_findings                 HEAD 2: lora_impression
  ==============================        ==============================
   Fokus: Deskripsi Anatomi              Fokus: Penalaran Klinis
   Input: Fitur Gambar                   Input: Fitur Gambar + Teks Findings
   Output: Teks "Findings"               Output: Teks "Impression"

## 1. Import Library and Set Reproducibility

In [ ]:
# =========================================================
# imports for training, vision-language modeling, and evaluation
# =========================================================

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from PIL import Image
import open_clip
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import evaluate
from bert_score import BERTScorer
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from pycocoevalcap.cider.cider import Cider
import warnings

# suppress warnings for cleaner logs
warnings.filterwarnings('ignore')


# =========================================================
# reproducibility setup
# =========================================================

def seed_everything(seed=42):
    # set python random seed
    import random
    random.seed(seed)

    # set numpy seed
    np.random.seed(seed)

    # set torch cpu seed
    torch.manual_seed(seed)

    # set torch gpu seed
    torch.cuda.manual_seed_all(seed)

    # enforce deterministic behavior (may reduce speed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # control python hash randomness
    os.environ['PYTHONHASHSEED'] = str(seed)


# apply global seed
seed_everything(42)

# set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"pipeline running on device: {device}")

pipeline running on device: cuda


## 2. Training Config

In [3]:
# =========================================================
# global configuration
# =========================================================

# dataset root directory
DATA_DIR = r"D:\VLM-Research_Task-C\data" 

# small language model configuration
LLM_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# vision encoder configuration
VISION_MODEL_NAME = "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224"

# maximum token length for text sequences
MAX_TXT_LEN = 256

# training batch size per iteration
BATCH_SIZE = 4           

# gradient accumulation steps
GRAD_ACCUM_STEPS = 4     

# optimizer learning rate
LEARNING_RATE = 2e-4

# total number of training epochs
EPOCHS = 3

# dataloader workers and pin memory
NUM_WORKERS = 0
PIN_MEMORY = True

# checkpoint save directory
CHECKPOINT_DIR = "D:/VLM-Research_Task-C/models"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [4]:
# =========================================================
# dataset and dataloader architecture
# =========================================================

class CheXpertPlusDualHeadDataset(Dataset):
    def __init__(self, csv_path, vision_processor, tokenizer, task="findings", max_txt_len=256):
        """
        dataset loader for decoupled conditional prompting
        task: "findings" or "impression"
        """
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.vision_processor = vision_processor
        self.tokenizer = tokenizer
        self.task = task
        self.max_txt_len = max_txt_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['actual_image_path']
        try:
            image = Image.open(img_path).convert("RGB")
            pixel_values = self.vision_processor(image)
        except Exception:
            pixel_values = torch.zeros(3, 224, 224)

        findings_text = str(row['cleaned_findings'])
        impression_text = str(row['cleaned_impression'])

        if self.task == "findings":
            prompt = (
                "<|im_start|>system\nYou are an expert radiologist. Describe the objective findings of this chest X-ray.<|im_end|>\n"
                "<|im_start|>user\n<tr><|im_end|>\n"
                "<|im_start|>assistant\n"
            )
            target_text = findings_text + "<|im_end|>"
        else:
            prompt = (
                "<|im_start|>system\nYou are an expert radiologist. Synthesize a definitive clinical impression based on the image and findings.<|im_end|>\n"
                f"<|im_start|>user\n<table>\nGiven Findings: {findings_text}<|im_end|>\n"
                "<|im_start|>assistant\n"
            )
            target_text = impression_text + "<|im_end|>"

        full_text = prompt + target_text
        tokens = self.tokenizer(
            full_text,
            max_length=self.max_txt_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        input_ids = tokens["input_ids"].squeeze(0)
        attention_mask = tokens["attention_mask"].squeeze(0)

        # autoregressive loss masking
        labels = input_ids.clone()
        prompt_len = len(self.tokenizer(prompt, truncation=True, max_length=self.max_txt_len)["input_ids"])
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "findings_text": findings_text,   # for metrics
            "impression_text": impression_text,
            "image_path": img_path
        }

In [ ]:
old = r"D:\VLM-Research_Task-C\data\ChexPert\vlm_fah"
new  = r"D:\VLM - Farhan Hamzah\vlm_fah"

# daftar file CSV (sesuaikan dengan nama file Anda)
csv_files = [
    "chexpert_train_split.csv",
    "chexpert_valid_split.csv",
    "chexpert_test_split.csv"
]

# lokasi penyimpanan CSV (sesuaikan dengan DATA_DIR Anda)
data_dir = r"D:\VLM - Farhan Hamzah\vlm_fah\dataset\chexpertplus\preprocessed\PNG"

for file in csv_files:
    path = os.path.join(data_dir, file)
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['actual_image_path'] = df['actual_image_path'].str.replace(old, new, regex=False)
        df.to_csv(path, index=False)
        print(f"✓ {file} diperbarui")
    else:
        print(f"✗ {file} tidak ditemukan di {path}")

In [ ]:
# =========================================================
# model initialization & semi-freeze vision backbone
# =========================================================

# initialize biomedclip vision encoder and preprocessing pipeline
vision_base_model, _, vision_processor = open_clip.create_model_and_transforms(VISION_MODEL_NAME)
vision_encoder = vision_base_model.visual

# freeze all vision encoder parameters
for param in vision_encoder.parameters():
    param.requires_grad = False

# unfreeze the last transformer block for chest x-ray domain adaptation
# transformer blocks in open_clip vit are stored in transformer.resblocks
if hasattr(vision_encoder, 'transformer'):
    for param in vision_encoder.transformer.resblocks[-1].parameters():
        param.requires_grad = True

print("-> semi-freeze successfully applied to the last vision transformer layer.")
vision_encoder.to(device)


# =========================================================
# qwen2.5-7b loading with 4-bit nf4 quantization
# =========================================================
os.environ['HF_HOME'] = 'D:/huggingface_cache'   # arahkan ke direktori D: (lab AI komputer) Hugging Face akan membuat direktori secara otomatis jika belum ada. Jadi aman.
print("[2/5] loading qwen2.5-7b in 4-bit nf4 quantization...")

# configure bitsandbytes 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# initialize tokenizer
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# load quantized llm with automatic device mapping
llm_base = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# qwen2.5-7b hidden embedding dimension
TEXT_EMBED_DIM = llm_base.config.hidden_size 


# =========================================================
# multimodal projector module
# =========================================================

class MedicalVLProjector(nn.Module):
    """project visual embeddings into qwen text embedding space"""

    def __init__(self, vision_dim=512, text_dim=3584):
        super().__init__()

        # two-layer projection head with gelu activation
        self.projector = nn.Sequential(
            nn.Linear(vision_dim, text_dim),
            nn.GELU(),
            nn.Linear(text_dim, text_dim)
        )

    def forward(self, x):
        return self.projector(x)


# initialize projector module
projector = MedicalVLProjector(
    vision_dim=512,
    text_dim=TEXT_EMBED_DIM
).to(device)

-> semi-freeze successfully applied to the last vision transformer layer.
[2/5] loading qwen2.5-7b in 4-bit nf4 quantization...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [ ]:
# =========================================================
#           dual-lora heads configuration
# =========================================================

print("[3/5] injecting decoupled dual-lora adapters...")

# define LoRA configuration for findings head
lora_config_findings = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# define LoRA configuration for impression head
lora_config_impression = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# inject first adapter (findings) into the base LLM model
llm_model = get_peft_model(llm_base, lora_config_findings, adapter_name="lora_findings")

# add second adapter (impression) to the same base model
llm_model.add_adapter("lora_impression", lora_config_impression)

print("-> dual-lora successfully registered into the model memory workspace.")

## 3. Train Loop

In [ ]:
# =========================================================
#      multi-task train loop (forward pass pipeline)
# =========================================================

def forward_step(batch, vision_encoder, projector, llm_model):

    # move batch tensors to target device
    pixel_values = batch["pixel_values"].to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].to(device)

    # =====================================================
    #  visual feature extraction
    # =====================================================

    # extract visual embeddings from chest x-ray images
    # autograd is active only for unfrozen visual layers
    img_features = vision_encoder(pixel_values) 

    # =====================================================
    #  visual-to-text projection
    # =====================================================

    # project visual embeddings into qwen embedding space
    # output shape: [batch_size, 1, hidden_dim]
    visual_tokens = projector(img_features).unsqueeze(1)

    # =====================================================
    #  text embedding extraction
    # =====================================================

    # retrieve token embeddings from qwen base model
    text_embeddings = llm_model.get_base_model().model.embed_tokens(input_ids)

    # =====================================================
    #  multimodal embedding fusion
    # =====================================================

    # prepend visual token before textual token embeddings
    inputs_embeds = torch.cat([visual_tokens, text_embeddings], dim=1)
    inputs_embeds = inputs_embeds.to(llm_model.dtype)

    # =====================================================
    #  attention mask and label alignment
    # =====================================================

    # adjust mask dimensions after adding visual token
    B, _ = input_ids.shape

    # create visual attention mask
    v_mask = torch.ones((B, 1), device=device)

    # extend original attention mask
    extended_attention_mask = torch.cat([v_mask, attention_mask], dim=1)

    # ignore visual token during loss computation
    v_label = torch.full((B, 1), -100, device=device)

    # extend labels with visual token placeholder
    extended_labels = torch.cat([v_label, labels], dim=1)

    # =====================================================
    #  causal language modeling forward pass
    # =====================================================

    # autoregressive token prediction
    outputs = llm_model(
        inputs_embeds=inputs_embeds,
        attention_mask=extended_attention_mask,
        labels=extended_labels
    )

    return outputs.loss

# =========================================================
# inference function for metrics
# =========================================================
def generate_report(batch, vision_encoder, projector, llm_model, adapter_name, max_new_tokens=128):
    llm_model.set_adapter(adapter_name)
    pixel_values = batch["pixel_values"].to(device)
    input_ids = batch["input_ids"].to(device)
    with torch.no_grad():
        img_features = vision_encoder(pixel_values)
        visual_tokens = projector(img_features).unsqueeze(1)
        text_embeddings = llm_model.get_base_model().model.embed_tokens(input_ids)
        inputs_embeds = torch.cat([visual_tokens, text_embeddings], dim=1)
        inputs_embeds = inputs_embeds.to(llm_model.dtype)
        generated_ids = llm_model.generate(
            inputs_embeds=inputs_embeds,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)



# =========================================================
# evaluation metrics computation function
# =========================================================
def compute_metrics(predictions, references):

    # predictions and references: list of generated and ground-truth strings
    smoothing = SmoothingFunction().method1

    # =====================================================
    # bleu score computation (1-gram to 4-gram)
    # =====================================================

    bleu_scores = {}

    for i in range(1, 5):

        bleu = corpus_bleu(
            [[r.split()] for r in references],
            [p.split() for p in predictions],
            weights=tuple([1.0 / i] * i),
            smoothing_function=smoothing
        )

        bleu_scores[f'bleu_{i}'] = bleu


    # =====================================================
    # rouge score computation
    # =====================================================

    rouge = evaluate.load('rouge')
    rouge_results = rouge.compute(
        predictions=predictions,
        references=references
    )


    # =====================================================
    # meteor score computation
    # =====================================================

    meteor = evaluate.load('meteor')
    meteor_score = meteor.compute(
        predictions=predictions,
        references=references
    )['meteor']


    # =====================================================
    # cider score computation
    # =====================================================

    cider_scorer = Cider()

    cider_score, _ = cider_scorer.compute_score(
        {i: [ref] for i, ref in enumerate(references)},
        {i: [pred] for i, pred in enumerate(predictions)}
    )


    # =====================================================
    # bertscore computation
    # =====================================================

    bert_scorer = BERTScorer(
        lang='en',
        rescale_with_baseline=True,
        device=device
    )

    _, _, f1 = bert_scorer.score(predictions, references)
    bert_f1 = f1.mean().item()


    # =====================================================
    # aggregate all metrics
    # =====================================================

    metrics = {
        **bleu_scores,
        'rouge1': rouge_results['rouge1'],
        'rouge2': rouge_results['rouge2'],
        'rougeL': rouge_results['rougeL'],
        'meteor': meteor_score,
        'cider': cider_score,
        'bertscore_f1': bert_f1
    }

    return metrics

# =========================================================
# dataset split configuration
# =========================================================

# define dataset split file paths
train_csv_file = os.path.join(DATA_DIR, "chexpert_train_split.csv")
val_csv_file = os.path.join(DATA_DIR, "chexpert_valid_split.csv")
test_csv_file = os.path.join(DATA_DIR, "chexpert_test_split.csv")


# =========================================================
# dataloader initialization for dual-head training
# =========================================================

# findings dataloaders
train_ds_findings = CheXpertPlusDualHeadDataset(train_csv_file, vision_processor, tokenizer, task="findings", max_txt_len=MAX_TXT_LEN)
val_ds_findings = CheXpertPlusDualHeadDataset(val_csv_file, vision_processor, tokenizer, task="findings", max_txt_len=MAX_TXT_LEN)
test_ds_findings  = CheXpertPlusDualHeadDataset(test_csv_file, vision_processor, tokenizer, "findings", MAX_TXT_LEN)
train_loader_findings = DataLoader(train_ds_findings, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader_findings = DataLoader(val_ds_findings, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader_findings  = DataLoader(test_ds_findings, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# impression dataloaders
train_ds_impression = CheXpertPlusDualHeadDataset(train_csv_file, vision_processor, tokenizer, task="impression", max_txt_len=MAX_TXT_LEN)
val_ds_impression = CheXpertPlusDualHeadDataset(val_csv_file, vision_processor, tokenizer, task="impression", max_txt_len=MAX_TXT_LEN)
test_ds_impression  = CheXpertPlusDualHeadDataset(test_csv_file, vision_processor, tokenizer, "impression", MAX_TXT_LEN)
train_loader_impression = DataLoader(train_ds_impression, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader_impression = DataLoader(val_ds_impression, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader_impression  = DataLoader(test_ds_impression, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# =========================================================
# optimizer configuration
# =========================================================

# collect all trainable parameters:
# - multimodal projector
# - unfrozen vision transformer layers
# - active lora adapter parameters
trainable_parameters = (
    list(projector.parameters()) + 
    [p for p in vision_encoder.parameters() if p.requires_grad] +
    [p for p in llm_model.parameters() if p.requires_grad]
)

# initialize adamw optimizer
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LEARNING_RATE
)

In [ ]:
# =========================================================
# multi-task dual-head training loop with validation
# =========================================================

print("\n[4/5] starting decoupled dual-head training with validation...")

# initialize loss tracking dictionary for monitoring training and validation curves
loss_history = {
    'train_findings': [],
    'train_impression': [],
    'val_findings': [],
    'val_impression': []
}

# track best validation loss for checkpoint selection
best_val_loss = float('inf')


for epoch in range(EPOCHS):

    # =====================================================
    # training findings head
    # =====================================================

    # activate findings adapter
    llm_model.set_adapter("lora_findings")

    # set all modules to training mode
    llm_model.train()
    vision_encoder.train()
    projector.train()

    epoch_loss_findings = 0.0
    optimizer.zero_grad()

    print(f"\n--- epoch {epoch+1}/{EPOCHS} | training findings head ---")

    for step, batch in enumerate(tqdm(train_loader_findings, desc="findings")):

        # forward pass and loss computation
        loss = forward_step(batch, vision_encoder, projector, llm_model)

        # normalize loss for gradient accumulation
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        # optimizer update step
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        # accumulate training loss
        epoch_loss_findings += loss.item() * GRAD_ACCUM_STEPS

    # compute average training loss for findings head
    avg_train_loss_f = epoch_loss_findings / len(train_loader_findings)
    loss_history['train_findings'].append(avg_train_loss_f)


    # =====================================================
    # training impression head
    # =====================================================

    # activate impression adapter
    llm_model.set_adapter("lora_impression")

    llm_model.train()

    epoch_loss_impression = 0.0
    optimizer.zero_grad()

    print(f"\n--- epoch {epoch+1}/{EPOCHS} | training impression head ---")

    for step, batch in enumerate(tqdm(train_loader_impression, desc="impression")):

        # forward pass and loss computation
        loss = forward_step(batch, vision_encoder, projector, llm_model)

        # normalize loss for gradient accumulation
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()

        # optimizer update step
        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad()

        # accumulate training loss
        epoch_loss_impression += loss.item() * GRAD_ACCUM_STEPS

    # compute average training loss for impression head
    avg_train_loss_i = epoch_loss_impression / len(train_loader_impression)
    loss_history['train_impression'].append(avg_train_loss_i)


    # =====================================================
    # validation loop
    # =====================================================

    # set evaluation mode for all modules
    llm_model.eval()
    vision_encoder.eval()
    projector.eval()

    val_loss_f = 0.0
    val_loss_i = 0.0

    # -----------------------------------------------------
    # validation for findings head
    # -----------------------------------------------------
    llm_model.set_adapter("lora_findings")
    with torch.no_grad():

        for batch in tqdm(val_loader_findings, desc="val findings"):

            # forward pass without gradient computation
            loss = forward_step(batch, vision_encoder, projector, llm_model)
            val_loss_f += loss.item()

        # average validation loss (findings)
        val_loss_f /= len(val_loader_findings)

    # -----------------------------------------------------
    # validation for impression head
    # -----------------------------------------------------

    # switch to impression adapter for evaluation
    llm_model.set_adapter("lora_impression")

    with torch.no_grad():

        for batch in tqdm(val_loader_impression, desc="val impression"):

            # forward pass without gradient computation
            loss = forward_step(batch, vision_encoder, projector, llm_model)
            val_loss_i += loss.item()

        # average validation loss (impression)
        val_loss_i /= len(val_loader_impression)


    # store validation losses
    loss_history['val_findings'].append(val_loss_f)
    loss_history['val_impression'].append(val_loss_i)


    # =====================================================
    # epoch logging
    # =====================================================
    print(
        f"\n>> epoch {epoch+1} | "
        f"train loss f: {avg_train_loss_f:.4f} i: {avg_train_loss_i:.4f} | "
        f"val loss f: {val_loss_f:.4f} i: {val_loss_i:.4f}"
    )


    # =====================================================
    # checkpointing (best validation model selection)
    # =====================================================

    # compute combined validation loss across both heads
    combined_val_loss = val_loss_f + val_loss_i

    # save best model checkpoint based on validation performance
    if combined_val_loss < best_val_loss:

        best_val_loss = combined_val_loss

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"best_model_epoch{epoch+1}.pt"
        )

        torch.save({
            'epoch': epoch,
            'projector_state': projector.state_dict(),
            'vision_encoder_state': vision_encoder.state_dict(),
            'llm_model_state': llm_model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_loss': combined_val_loss,
        }, checkpoint_path)

        print(f"-> checkpoint saved: {checkpoint_path}")


    # =====================================================
    # save latest epoch checkpoint
    # =====================================================

    torch.save({
        'epoch': epoch,
        'projector_state': projector.state_dict(),
        'vision_encoder_state': vision_encoder.state_dict(),
        'llm_model_state': llm_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
    }, os.path.join(CHECKPOINT_DIR, f"last_epoch{epoch+1}.pt"))

## 4. Inference and Evaluation

In [ ]:
# =========================================================
# loss curve visualization
# =========================================================
plt.figure(figsize=(10,5))
plt.plot(loss_history['train_findings'], label='train findings')
plt.plot(loss_history['train_impression'], label='train impression')
plt.plot(loss_history['val_findings'], label='val findings')
plt.plot(loss_history['val_impression'], label='val impression')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.title('training and validation loss curves')
plt.legend()
plt.grid(True)
plt.savefig('loss_curves.png')
plt.show()
print("-> loss curves saved as loss_curves.png")

In [ ]:
# =========================================================
# full evaluation on test set + export csv
# =========================================================

print("\n[5/5] full evaluation on test set...")
llm_model.eval()
vision_encoder.eval()
projector.eval()

# lists to store results
img_paths = []
pred_f, ref_f = [], []
pred_i, ref_i = [], []

# evaluate findings head
for batch in tqdm(test_loader_findings, desc="test findings"):
    pred = generate_report(batch, vision_encoder, projector, llm_model, "lora_findings")
    pred_f.append(pred)
    ref_f.append(batch['findings_text'][0])
    img_paths.append(batch['image_path'][0])   # get image path

# evaluate impression head
for batch in tqdm(test_loader_impression, desc="test impression"):
    pred = generate_report(batch, vision_encoder, projector, llm_model, "lora_impression")
    pred_i.append(pred)
    ref_i.append(batch['impression_text'][0])

# compute metrics (optional, tetap lakukan)
print("\ncomputing metrics for findings...")
metrics_f = compute_metrics(pred_f, ref_f)
print("\ncomputing metrics for impression...")
metrics_i = compute_metrics(pred_i, ref_i)

# print metrics
print("\n" + "="*60)
print("final test set evaluation metrics")
print("="*60)
print("\n[findings head]")
for k, v in metrics_f.items():
    print(f"  {k}: {v:.4f}")
print("\n[impression head]")
for k, v in metrics_i.items():
    print(f"  {k}: {v:.4f}")
print("="*60)

# =========================================================
# export to csv
# =========================================================
export_df = pd.DataFrame({
    'image_path': img_paths,
    'ground_truth_findings': ref_f,
    'predicted_findings': pred_f,
    'ground_truth_impression': ref_i,
    'predicted_impression': pred_i
})
output_csv = os.path.join(DATA_DIR, "test_predictions.csv")
export_df.to_csv(output_csv, index=False)
print(f"\n-> predictions exported to {output_csv}")

# optional sample
print("\nsample generated findings:")
print(pred_f[0] if pred_f else "none")
print("\nsample generated impression:")
print(pred_i[0] if pred_i else "none")

print("\npipeline complete.")